In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Pipeline Completo: SQL Server para Landing Zone (Excel) - Com Limpeza

# COMMAND ----------

# Instalar dependências necessárias
%pip install azure-storage-file-datalake

# COMMAND ----------

# Importar bibliotecas
import pandas as pd
from azure.storage.filedatalake import DataLakeServiceClient
from azure.core.exceptions import ResourceExistsError, ResourceNotFoundError
import io
from datetime import datetime

# COMMAND ----------

# Configurações - SUBSTITUA PELOS SEUS VALORES
# Azure Data Lake Storage
ADLS_ACCOUNT_NAME = "datalake7eadf73a479de9f7"
ADLS_FILE_SYSTEM_NAME = "landing-zone"
ADLS_DIRECTORY_NAME = "excel-exports"
ADLS_SAS_TOKEN = "sv=2024-11-04&ss=bfqt&srt=sco&sp=rwdlacupyx&se=2025-06-21T09:26:44Z&st=2025-06-21T01:26:44Z&spr=https&sig=WOCAHd1wohTvXLL14dha1ny5HxIwk4YM3TNgR2vJ5yA%3D"

# SQL Server - SUBSTITUA PELOS SEUS VALORES
SQL_SERVER_HOST = "sql-pipeline-sexta.database.windows.net"
SQL_SERVER_PORT = "1433"
SQL_DATABASE = "dados"
SQL_SERVER_USER = "admin_user"  # SUBSTITUA
SQL_SERVER_PASSWORD = "senha@2022"  # SUBSTITUA

# JDBC URL
jdbc_url = f"jdbc:sqlserver://{SQL_SERVER_HOST}:{SQL_SERVER_PORT};database={SQL_DATABASE}"

print("✅ Configurações carregadas:")
print(f"   SQL Server: {SQL_SERVER_HOST}")
print(f"   Database: {SQL_DATABASE}")
print(f"   Azure Storage: {ADLS_ACCOUNT_NAME}")
print(f"   File System: {ADLS_FILE_SYSTEM_NAME}")

# COMMAND ----------

def clean_landing_zone():
    """
    Remove todos os arquivos existentes no diretório da landing zone usando dbutils
    """
    try:
        print(f"🗑️ Limpando diretório da landing zone: {ADLS_DIRECTORY_NAME}")
        
        # Path usando dbutils
        landing_path = f"/mnt/{ADLS_ACCOUNT_NAME}/{ADLS_FILE_SYSTEM_NAME}/{ADLS_DIRECTORY_NAME}"
        
        try:
            # Verificar se o path existe e listar arquivos
            files_and_dirs = dbutils.fs.ls(landing_path)
        except Exception:
            print(f"   ℹ️ Diretório {landing_path} não existe ou está vazio - será criado durante o processo")
            return True
        
        if not files_and_dirs:
            print("   ℹ️ Diretório já está vazio")
            return True
        
        print(f"   📋 Encontrados {len(files_and_dirs)} itens para exclusão:")
        
        deleted_count = 0
        error_count = 0
        
        # Excluir cada item (arquivos e diretórios)
        for item in files_and_dirs:
            try:
                dbutils.fs.rm(item.path, True)  # True para recursivo
                deleted_count += 1
                print(f"   ✅ Excluído: {item.name}")
            except Exception as e:
                error_count += 1
                print(f"   ❌ Erro ao excluir {item.name}: {e}")
        
        print(f"   📊 Resumo da limpeza:")
        print(f"      ✅ Itens excluídos: {deleted_count}")
        print(f"      ❌ Erros: {error_count}")
        
        return error_count == 0
        
    except Exception as e:
        print(f"   ❌ Erro geral durante limpeza da landing zone: {e}")
        return False

# COMMAND ----------

def create_csv_for_table(table_name, df_spark):
    """
    Cria um arquivo CSV para uma tabela específica
    """
    try:
        print(f"   🔄 Convertendo Spark DataFrame para Pandas...")
        
        # Converter Spark DataFrame para Pandas
        df_pandas = df_spark.toPandas()
        
        print(f"   📝 Criando arquivo CSV...")
        
        # Nome do arquivo
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        csv_filename = f"{table_name}_{timestamp}.csv"
        
        # Converter DataFrame para CSV em memória
        csv_buffer = io.StringIO()
        df_pandas.to_csv(csv_buffer, index=False, encoding='utf-8', sep=',')
        csv_data = csv_buffer.getvalue()
        
        print(f"   ✅ CSV criado: {csv_filename}")
        return csv_data, csv_filename
        
    except Exception as e:
        print(f"   ❌ Erro ao criar CSV para {table_name}: {e}")
        return None, None

# COMMAND ----------

def save_csv_to_landing(csv_data, csv_filename):
    """
    Salva o arquivo CSV no Azure Data Lake Storage
    """
    try:
        print(f"   🔄 Salvando {csv_filename} no Azure Data Lake...")
        
        # Criar cliente do Azure Data Lake
        service_client = DataLakeServiceClient(
            account_url=f"https://{ADLS_ACCOUNT_NAME}.dfs.core.windows.net",
            credential=ADLS_SAS_TOKEN
        )
        
        file_system_client = service_client.get_file_system_client(ADLS_FILE_SYSTEM_NAME)
        
        # Criar diretório se não existir
        try:
            directory_client = file_system_client.get_directory_client(ADLS_DIRECTORY_NAME)
            directory_client.create_directory()
        except ResourceExistsError:
            directory_client = file_system_client.get_directory_client(ADLS_DIRECTORY_NAME)
        
        # Upload para Azure Data Lake
        file_client = directory_client.get_file_client(csv_filename)
        file_client.upload_data(csv_data.encode('utf-8'), overwrite=True)
        
        print(f"   ✅ Arquivo salvo: {ADLS_DIRECTORY_NAME}/{csv_filename}")
        return True
        
    except Exception as e:
        print(f"   ❌ Erro ao salvar {csv_filename}: {e}")
        return False

# COMMAND ----------

# ETAPA 1: LIMPEZA DA LANDING ZONE
print("🚀 INICIANDO PIPELINE - ETAPA 1: LIMPEZA")
print("=" * 60)

if clean_landing_zone():
    print("✅ Limpeza da landing zone concluída com sucesso!")
else:
    print("⚠️ Houve problemas na limpeza, mas continuando com o processo...")

print("\n" + "=" * 60)

# COMMAND ----------

# ETAPA 2: OBTER LISTA DE TABELAS
print("🚀 ETAPA 2: OBTENDO LISTA DE TABELAS")
print("=" * 60)

try:
    # Usando query de consulta direta
    query = """
    SELECT TABLE_SCHEMA, TABLE_NAME 
    FROM INFORMATION_SCHEMA.TABLES 
    WHERE TABLE_TYPE = 'BASE TABLE'
    """
    
    tables_df = spark.read.format("jdbc") \
        .option("url", jdbc_url) \
        .option("query", query) \
        .option("user", SQL_SERVER_USER) \
        .option("password", SQL_SERVER_PASSWORD) \
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .load()
    
    table_names = [row.TABLE_NAME for row in tables_df.collect()]
    print(f"✅ Tabelas encontradas ({len(table_names)}): {table_names}")
    
except Exception as e:
    print(f"❌ Erro ao obter nomes das tabelas com query: {e}")
    
    # Fallback: Tentar com abordagem alternativa
    try:
        print("🔄 Tentando abordagem alternativa...")
        connection_properties = {
            "user": SQL_SERVER_USER,
            "password": SQL_SERVER_PASSWORD,
            "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
        }
        
        tables_df = spark.read.jdbc(
            url=jdbc_url,
            table="INFORMATION_SCHEMA.TABLES",
            properties=connection_properties
        ).filter("TABLE_TYPE = 'BASE TABLE'")
        
        table_names = [row.TABLE_NAME for row in tables_df.collect()]
        print(f"✅ Tabelas encontradas com fallback ({len(table_names)}): {table_names}")
        
    except Exception as e2:
        print(f"❌ Erro também na abordagem alternativa: {e2}")
        table_names = []

print("\n" + "=" * 60)

# COMMAND ----------

# ETAPA 3: PROCESSAMENTO DAS TABELAS
print("🚀 ETAPA 3: PROCESSANDO TABELAS")
print("=" * 60)

if table_names:
    print(f"📊 Iniciando processamento de {len(table_names)} tabelas...")
    
    success_count = 0
    error_count = 0
    
    for i, table_name in enumerate(table_names, 1):
        print(f"\n📋 Processando tabela {i}/{len(table_names)}: {table_name}")
        
        try:
            # Ler dados da tabela
            df_spark = spark.read.format("jdbc") \
                .option("url", jdbc_url) \
                .option("query", f"SELECT * FROM dbo.{table_name}") \
                .option("user", SQL_SERVER_USER) \
                .option("password", SQL_SERVER_PASSWORD) \
                .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
                .load()
            
            # Verificar se a tabela tem dados
            row_count = df_spark.count()
            print(f"   📈 Registros encontrados: {row_count}")
            
            if row_count == 0:
                print(f"   ⚠️ Tabela {table_name} está vazia. Pulando...")
                continue
            
            # Criar CSV para esta tabela
            csv_data, csv_filename = create_csv_for_table(table_name, df_spark)
            
            if csv_data and csv_filename:
                # Salvar na landing zone
                if save_csv_to_landing(csv_data, csv_filename):
                    success_count += 1
                    print(f"   ✅ Sucesso: {csv_filename}")
                else:
                    error_count += 1
                    print(f"   ❌ Falha ao salvar: {table_name}")
            else:
                error_count += 1
                print(f"   ❌ Falha ao criar CSV: {table_name}")
                
        except Exception as e:
            error_count += 1
            print(f"   ❌ Erro ao processar tabela '{table_name}': {e}")
    
    print("\n" + "=" * 60)
    print("🏁 RESUMO FINAL DO PIPELINE")
    print("=" * 60)
    print(f"   ✅ Sucessos: {success_count}")
    print(f"   ❌ Erros: {error_count}")
    print(f"   📊 Total processado: {success_count + error_count}/{len(table_names)}")
    
else:
    print("❌ Nenhuma tabela encontrada para processar.")

print("\n🎉 PIPELINE CONCLUÍDO!")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Verificação dos arquivos criados

# COMMAND ----------

def list_created_files():
    """
    Lista os arquivos Excel criados no Azure Data Lake usando dbutils
    """
    try:
        print(f"📁 Listando arquivos no diretório: {ADLS_DIRECTORY_NAME}")
        
        # Path usando dbutils
        landing_path = f"/mnt/{ADLS_ACCOUNT_NAME}/{ADLS_FILE_SYSTEM_NAME}/{ADLS_DIRECTORY_NAME}"
        
        try:
            files_and_dirs = dbutils.fs.ls(landing_path)
        except Exception:
            print(f"   ℹ️ Diretório {landing_path} não existe ou está vazio")
            return
        
        excel_files = []
        for item in files_and_dirs:
            if not item.name.endswith('/') and (item.name.endswith('.xlsx') or item.name.endswith('.csv')):
                excel_files.append(item.name)
        
        if excel_files:
            print(f"📋 Arquivos encontrados ({len(excel_files)}):")
            for file in sorted(excel_files):
                print(f"   📄 {file}")
        else:
            print("   ℹ️ Nenhum arquivo Excel encontrado")
            
    except Exception as e:
        print(f"❌ Erro ao listar arquivos: {e}")

# Executar listagem
list_created_files()

# COMMAND ----------

# MAGIC %md
# MAGIC ## Teste de Conexão (Opcional)

# COMMAND ----------

def test_connection():
    """
    Testa a conexão com SQL Server e Azure Data Lake
    """
    print("🔍 Testando conexões...")
    
    # Teste SQL Server
    try:
        test_df = spark.read.format("jdbc") \
            .option("url", jdbc_url) \
            .option("query", "SELECT 1 as test") \
            .option("user", SQL_SERVER_USER) \
            .option("password", SQL_SERVER_PASSWORD) \
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
            .load()
        
        print("✅ Conexão SQL Server: OK")
        
    except Exception as e:
        print(f"❌ Conexão SQL Server: FALHA - {e}")
    
    # Teste Azure Data Lake
    try:
        service_client = DataLakeServiceClient(
            account_url=f"https://{ADLS_ACCOUNT_NAME}.dfs.core.windows.net",
            credential=ADLS_SAS_TOKEN
        )
        
        file_system_client = service_client.get_file_system_client(ADLS_FILE_SYSTEM_NAME)
        file_system_client.get_file_system_properties()
        
        print("✅ Conexão Azure Data Lake: OK")
        
    except Exception as e:
        print(f"❌ Conexão Azure Data Lake: FALHA - {e}")

# Descomente a linha abaixo para testar as conexões
# test_connection()# Databricks notebook source
# MAGIC %md
# MAGIC # Pipeline Completo: SQL Server para Landing Zone (Excel) - Com Limpeza

# COMMAND ----------

# Instalar dependências necessárias
%pip install azure-storage-file-datalake

# COMMAND ----------

# Importar bibliotecas
import pandas as pd
from azure.storage.filedatalake import DataLakeServiceClient
from azure.core.exceptions import ResourceExistsError, ResourceNotFoundError
import io
from datetime import datetime

# COMMAND ----------

# Configurações - SUBSTITUA PELOS SEUS VALORES
# Azure Data Lake Storage
ADLS_ACCOUNT_NAME = "datalake7eadf73a479de9f7"
ADLS_FILE_SYSTEM_NAME = "landing-zone"
ADLS_DIRECTORY_NAME = "excel-exports"
ADLS_SAS_TOKEN = "sv=2024-11-04&ss=bfqt&srt=sco&sp=rwdlacupyx&se=2025-06-21T09:26:44Z&st=2025-06-21T01:26:44Z&spr=https&sig=WOCAHd1wohTvXLL14dha1ny5HxIwk4YM3TNgR2vJ5yA%3D"

# SQL Server - SUBSTITUA PELOS SEUS VALORES
SQL_SERVER_HOST = "sql-pipeline-sexta.database.windows.net"
SQL_SERVER_PORT = "1433"
SQL_DATABASE = "dados"
SQL_SERVER_USER = "admin_user"  # SUBSTITUA
SQL_SERVER_PASSWORD = "senha@2022"  # SUBSTITUA

# JDBC URL
jdbc_url = f"jdbc:sqlserver://{SQL_SERVER_HOST}:{SQL_SERVER_PORT};database={SQL_DATABASE}"

print("✅ Configurações carregadas:")
print(f"   SQL Server: {SQL_SERVER_HOST}")
print(f"   Database: {SQL_DATABASE}")
print(f"   Azure Storage: {ADLS_ACCOUNT_NAME}")
print(f"   File System: {ADLS_FILE_SYSTEM_NAME}")

# COMMAND ----------

def clean_landing_zone():
    """
    Remove todos os arquivos existentes no diretório da landing zone usando dbutils
    """
    try:
        print(f"🗑️ Limpando diretório da landing zone: {ADLS_DIRECTORY_NAME}")
        
        # Path usando dbutils
        landing_path = f"/mnt/{ADLS_ACCOUNT_NAME}/{ADLS_FILE_SYSTEM_NAME}/{ADLS_DIRECTORY_NAME}"
        
        try:
            # Verificar se o path existe e listar arquivos
            files_and_dirs = dbutils.fs.ls(landing_path)
        except Exception:
            print(f"   ℹ️ Diretório {landing_path} não existe ou está vazio - será criado durante o processo")
            return True
        
        if not files_and_dirs:
            print("   ℹ️ Diretório já está vazio")
            return True
        
        print(f"   📋 Encontrados {len(files_and_dirs)} itens para exclusão:")
        
        deleted_count = 0
        error_count = 0
        
        # Excluir cada item (arquivos e diretórios)
        for item in files_and_dirs:
            try:
                dbutils.fs.rm(item.path, True)  # True para recursivo
                deleted_count += 1
                print(f"   ✅ Excluído: {item.name}")
            except Exception as e:
                error_count += 1
                print(f"   ❌ Erro ao excluir {item.name}: {e}")
        
        print(f"   📊 Resumo da limpeza:")
        print(f"      ✅ Itens excluídos: {deleted_count}")
        print(f"      ❌ Erros: {error_count}")
        
        return error_count == 0
        
    except Exception as e:
        print(f"   ❌ Erro geral durante limpeza da landing zone: {e}")
        return False

# COMMAND ----------

def create_csv_for_table(table_name, df_spark):
    """
    Cria um arquivo CSV para uma tabela específica
    """
    try:
        print(f"   🔄 Convertendo Spark DataFrame para Pandas...")
        
        # Converter Spark DataFrame para Pandas
        df_pandas = df_spark.toPandas()
        
        print(f"   📝 Criando arquivo CSV...")
        
        # Nome do arquivo
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        csv_filename = f"{table_name}_{timestamp}.csv"
        
        # Converter DataFrame para CSV em memória
        csv_buffer = io.StringIO()
        df_pandas.to_csv(csv_buffer, index=False, encoding='utf-8', sep=',')
        csv_data = csv_buffer.getvalue()
        
        print(f"   ✅ CSV criado: {csv_filename}")
        return csv_data, csv_filename
        
    except Exception as e:
        print(f"   ❌ Erro ao criar CSV para {table_name}: {e}")
        return None, None

# COMMAND ----------

def save_csv_to_landing(csv_data, csv_filename):
    """
    Salva o arquivo CSV no Azure Data Lake Storage
    """
    try:
        print(f"   🔄 Salvando {csv_filename} no Azure Data Lake...")
        
        # Criar cliente do Azure Data Lake
        service_client = DataLakeServiceClient(
            account_url=f"https://{ADLS_ACCOUNT_NAME}.dfs.core.windows.net",
            credential=ADLS_SAS_TOKEN
        )
        
        file_system_client = service_client.get_file_system_client(ADLS_FILE_SYSTEM_NAME)
        
        # Criar diretório se não existir
        try:
            directory_client = file_system_client.get_directory_client(ADLS_DIRECTORY_NAME)
            directory_client.create_directory()
        except ResourceExistsError:
            directory_client = file_system_client.get_directory_client(ADLS_DIRECTORY_NAME)
        
        # Upload para Azure Data Lake
        file_client = directory_client.get_file_client(csv_filename)
        file_client.upload_data(csv_data.encode('utf-8'), overwrite=True)
        
        print(f"   ✅ Arquivo salvo: {ADLS_DIRECTORY_NAME}/{csv_filename}")
        return True
        
    except Exception as e:
        print(f"   ❌ Erro ao salvar {csv_filename}: {e}")
        return False

# COMMAND ----------

# ETAPA 1: LIMPEZA DA LANDING ZONE
print("🚀 INICIANDO PIPELINE - ETAPA 1: LIMPEZA")
print("=" * 60)

if clean_landing_zone():
    print("✅ Limpeza da landing zone concluída com sucesso!")
else:
    print("⚠️ Houve problemas na limpeza, mas continuando com o processo...")

print("\n" + "=" * 60)

# COMMAND ----------

# ETAPA 2: OBTER LISTA DE TABELAS
print("🚀 ETAPA 2: OBTENDO LISTA DE TABELAS")
print("=" * 60)

try:
    # Usando query de consulta direta
    query = """
    SELECT TABLE_SCHEMA, TABLE_NAME 
    FROM INFORMATION_SCHEMA.TABLES 
    WHERE TABLE_TYPE = 'BASE TABLE'
    """
    
    tables_df = spark.read.format("jdbc") \
        .option("url", jdbc_url) \
        .option("query", query) \
        .option("user", SQL_SERVER_USER) \
        .option("password", SQL_SERVER_PASSWORD) \
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .load()
    
    table_names = [row.TABLE_NAME for row in tables_df.collect()]
    print(f"✅ Tabelas encontradas ({len(table_names)}): {table_names}")
    
except Exception as e:
    print(f"❌ Erro ao obter nomes das tabelas com query: {e}")
    
    # Fallback: Tentar com abordagem alternativa
    try:
        print("🔄 Tentando abordagem alternativa...")
        connection_properties = {
            "user": SQL_SERVER_USER,
            "password": SQL_SERVER_PASSWORD,
            "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
        }
        
        tables_df = spark.read.jdbc(
            url=jdbc_url,
            table="INFORMATION_SCHEMA.TABLES",
            properties=connection_properties
        ).filter("TABLE_TYPE = 'BASE TABLE'")
        
        table_names = [row.TABLE_NAME for row in tables_df.collect()]
        print(f"✅ Tabelas encontradas com fallback ({len(table_names)}): {table_names}")
        
    except Exception as e2:
        print(f"❌ Erro também na abordagem alternativa: {e2}")
        table_names = []

print("\n" + "=" * 60)

# COMMAND ----------

# ETAPA 3: PROCESSAMENTO DAS TABELAS
print("🚀 ETAPA 3: PROCESSANDO TABELAS")
print("=" * 60)

if table_names:
    print(f"📊 Iniciando processamento de {len(table_names)} tabelas...")
    
    success_count = 0
    error_count = 0
    
    for i, table_name in enumerate(table_names, 1):
        print(f"\n📋 Processando tabela {i}/{len(table_names)}: {table_name}")
        
        try:
            # Ler dados da tabela
            df_spark = spark.read.format("jdbc") \
                .option("url", jdbc_url) \
                .option("query", f"SELECT * FROM dbo.{table_name}") \
                .option("user", SQL_SERVER_USER) \
                .option("password", SQL_SERVER_PASSWORD) \
                .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
                .load()
            
            # Verificar se a tabela tem dados
            row_count = df_spark.count()
            print(f"   📈 Registros encontrados: {row_count}")
            
            if row_count == 0:
                print(f"   ⚠️ Tabela {table_name} está vazia. Pulando...")
                continue
            
            # Criar CSV para esta tabela
            csv_data, csv_filename = create_csv_for_table(table_name, df_spark)
            
            if csv_data and csv_filename:
                # Salvar na landing zone
                if save_csv_to_landing(csv_data, csv_filename):
                    success_count += 1
                    print(f"   ✅ Sucesso: {csv_filename}")
                else:
                    error_count += 1
                    print(f"   ❌ Falha ao salvar: {table_name}")
            else:
                error_count += 1
                print(f"   ❌ Falha ao criar CSV: {table_name}")
                
        except Exception as e:
            error_count += 1
            print(f"   ❌ Erro ao processar tabela '{table_name}': {e}")
    
    print("\n" + "=" * 60)
    print("🏁 RESUMO FINAL DO PIPELINE")
    print("=" * 60)
    print(f"   ✅ Sucessos: {success_count}")
    print(f"   ❌ Erros: {error_count}")
    print(f"   📊 Total processado: {success_count + error_count}/{len(table_names)}")
    
else:
    print("❌ Nenhuma tabela encontrada para processar.")

print("\n🎉 PIPELINE CONCLUÍDO!")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Verificação dos arquivos criados

# COMMAND ----------

def list_created_files():
    """
    Lista os arquivos Excel criados no Azure Data Lake usando dbutils
    """
    try:
        print(f"📁 Listando arquivos no diretório: {ADLS_DIRECTORY_NAME}")
        
        # Path usando dbutils
        landing_path = f"/mnt/{ADLS_ACCOUNT_NAME}/{ADLS_FILE_SYSTEM_NAME}/{ADLS_DIRECTORY_NAME}"
        
        try:
            files_and_dirs = dbutils.fs.ls(landing_path)
        except Exception:
            print(f"   ℹ️ Diretório {landing_path} não existe ou está vazio")
            return
        
        excel_files = []
        for item in files_and_dirs:
            if not item.name.endswith('/') and (item.name.endswith('.xlsx') or item.name.endswith('.csv')):
                excel_files.append(item.name)
        
        if excel_files:
            print(f"📋 Arquivos encontrados ({len(excel_files)}):")
            for file in sorted(excel_files):
                print(f"   📄 {file}")
        else:
            print("   ℹ️ Nenhum arquivo Excel encontrado")
            
    except Exception as e:
        print(f"❌ Erro ao listar arquivos: {e}")

# Executar listagem
list_created_files()

# COMMAND ----------

# MAGIC %md
# MAGIC ## Teste de Conexão (Opcional)

# COMMAND ----------

def test_connection():
    """
    Testa a conexão com SQL Server e Azure Data Lake
    """
    print("🔍 Testando conexões...")
    
    # Teste SQL Server
    try:
        test_df = spark.read.format("jdbc") \
            .option("url", jdbc_url) \
            .option("query", "SELECT 1 as test") \
            .option("user", SQL_SERVER_USER) \
            .option("password", SQL_SERVER_PASSWORD) \
            .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
            .load()
        
        print("✅ Conexão SQL Server: OK")
        
    except Exception as e:
        print(f"❌ Conexão SQL Server: FALHA - {e}")
    
    # Teste Azure Data Lake
    try:
        service_client = DataLakeServiceClient(
            account_url=f"https://{ADLS_ACCOUNT_NAME}.dfs.core.windows.net",
            credential=ADLS_SAS_TOKEN
        )
        
        file_system_client = service_client.get_file_system_client(ADLS_FILE_SYSTEM_NAME)
        file_system_client.get_file_system_properties()
        
        print("✅ Conexão Azure Data Lake: OK")
        
    except Exception as e:
        print(f"❌ Conexão Azure Data Lake: FALHA - {e}")

# Descomente a linha abaixo para testar as conexões
# test_connection()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✅ Configurações carregadas:
   SQL Server: sql-pipeline-sexta.database.windows.net
   Database: dados
   Azure Storage: datalake7eadf73a479de9f7
   File System: landing-zone
🚀 INICIANDO PIPELINE - ETAPA 1: LIMPEZA
🗑️ Limpando diretório da landing zone: excel-exports
   📋 Encontrados 15 itens para exclusão:
   ✅ Excluído: _prisma_migrations_20250621_014253.xlsx
   ✅ Excluído: achievement_unlocked_20250621_014327.xlsx
   ✅ Excluído: achievements_20250621_014323.xlsx
   ✅ Excluído: developers_20250621_014303.xlsx
   ✅ Excluído: dlcs_20250621_014330.xlsx
   ✅ Excluído: game_genders_20250621_014334.xlsx
   ✅ Excluído: game_platforms_20250621_014340.xlsx
   ✅ Excluído: game_tags_20250621_014337.xlsx
   ✅ Excluído: games_20250621_014300.xlsx
   ✅ Excluído: genders_20250621_014310.xlsx
   ✅ Excluído: platforms_20250621_014306.xlsx
   ✅ Excluído: purchases_20250621_014320.x